<a href="https://colab.research.google.com/github/Aksinhaa/ColabFold/blob/main/TGW_pca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This workshop is a joint collaboration between NCBS - Bangalore, India `(Uma Ramakrishnan, Laura Bertola, Devkant Singha, Vinti Nanda)`; University of Edinburgh UK `(Katerina Guschanski, German Hernandez Alonso)`; IISc - Bangalore, India `(Anubhab Khan, Akancha Sinha )`; IISER Mohali, India `(Kritika M. Garg)`; and Birbal Sahni Institute of Palaeosciences (BSIP) - Lucknow, India `(Niraj Rai)`.
Scripts written by: `German Hernandez Alonso`
Notebook compiled by: `Akancha Sinha`

 # Population Structure Analysis using PCA (PLINK)

One of the main objectives of population genomics is to comprehend the genetic links between people and populations. Principal Component Analysis (PCA), a dimensionality reduction technique that reduces high-dimensional genotype data into a smaller number of components that capture the primary axes of genetic variation, is one of the most popular methods for examining such interactions.In genomic datasets, each individual is characterized by thousands to millions of single nucleotide polymorphisms (SNPs).  Such high-dimensional data cannot be directly interpreted. By projecting individuals onto a low-dimensional space, where genetically similar individuals cluster together and genetically dissimilar groups divide along principle components, PCA reduces this complexity.


##  Data Processing Before PCA

Before performing PCA, it is critical to ensure that the dataset is of high quality and free from biases. Two important preprocessing steps are applied:

### 1. Minor Allele Frequency (MAF) Filtering

Rare variants (low-frequency SNPs) can introduce noise and reduce the robustness of PCA. Therefore, SNPs with a minor allele frequency below 1% are removed.

### 2. Linkage Disequilibrium (LD) Pruning

Many SNPs in the genome are correlated due to physical linkage. Including correlated SNPs can bias PCA results. LD pruning removes such correlated variants, ensuring that only approximately independent SNPs are retained.

---

##  Tools Used

The analysis is performed using PLINK, a widely used toolset for genome-wide association studies and population genetics. Visualization is carried out in R using packages such as **ggplot2** and **ggrepel**.

---



This notebook performs:

* SNP filtering based on minor allele frequency
* Linkage disequilibrium pruning
* Principal Component Analysis (PCA)
* Visualization of genetic clustering across pigeon populations



In [ ]:
%%bash
# Install Miniconda
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda

# Initialize conda
source /usr/local/miniconda/etc/profile.d/conda.sh

# Accept Terms of Service
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Add channels
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda create -y -n ngs_env plink
R -e "install.packages(c('tidyverse','ggplot2','ggrepel'), repos='http://cran.rstudio.com/')"

In [ ]:
%%bash

mkdir -p /content/pca_plink
cd /content/pca_plink

In [ ]:
%%bash

cd /content/pca_plink

wget -nc https://zenodo.org/records/19914378/files/all_pigeons_newIDs-onlyTrans-mf01-noOut-geno.bed
wget -nc https://zenodo.org/records/19914378/files/all_pigeons_newIDs-onlyTrans-mf01-noOut-geno.bim
wget -nc https://zenodo.org/records/19914378/files/all_pigeons_newIDs-onlyTrans-mf01-noOut-geno.fam

In [ ]:
%%bash
cd /content/pca_plink

cp all_pigeons_newIDs-onlyTrans-mf01-noOut-geno.bed Pigeons_snp.bed
cp all_pigeons_newIDs-onlyTrans-mf01-noOut-geno.bim Pigeons_snp.bim
cp all_pigeons_newIDs-onlyTrans-mf01-noOut-geno.fam Pigeons_snp.fam

This script activates a Conda environment named Conda (ngs_env) to ensure all required bioinformatics tools are available. It then navigates to the directory `/content/pca_plink `where the genotype data files are stored. The PLINK command filters SNP data from the binary dataset Pigeons_snp, removing variants with a minor allele frequency (MAF) less than 0.01 and generating a new binary dataset. Finally, the filtered dataset is saved as `Pigeons_snp-mf01`.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/pca_plink

plink \
  --bfile Pigeons_snp \
  --maf 0.01 \
  --biallelic-only \
  --make-bed \
  --out Pigeons_snp-mf01 \
  --allow-extra-chr

This step performs linkage disequilibrium (LD) pruning using PLINK to remove correlated SNPs from the dataset. The `--indep-pairwise 50 10 0.4 `option scans SNPs in windows of 50 variants, shifts the window by 10 SNPs, and removes one SNP from pairs with high correlation (r² > 0.4).

This is important because many nearby SNPs are not independent due to linkage, and including them can bias downstream analyses like PCA. By retaining only relatively independent variants, LD pruning reduces redundancy and noise in the dataset.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env
cd /content/pca_plink

plink \
  --bfile Pigeons_snp-mf01 \
  --indep-pairwise 50 10 0.4 \
  --out LD_prune \
  --allow-extra-chr

This step uses PLINK to create a new dataset containing only the SNPs that passed linkage disequilibrium (LD) pruning. The `--extract LD_prune.prune.in` option selects the list of independent SNPs identified in the previous step. The `--make-bed` command generates a new set of binary PLINK files with these filtered variants, saved as Pigeons_snp_mf01_LDpruned.

This step is important because it ensures that only uncorrelated, high-quality SNPs are used for downstream analyses like PCA, improving accuracy and reducing bias.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env
cd /content/pca_plink

plink \
  --bfile Pigeons_snp-mf01 \
  --extract LD_prune.prune.in \
  --make-bed \
  --out Pigeons_snp-mf01-LDpruned \
  --allow-extra-chr

This step performs Principal Component Analysis (PCA) using PLINK on the LD-pruned SNP dataset.

 The `--bfile Pigeons_snp_mf01_LDpruned` option inputs the filtered, independent variants, and `--pca 10 `computes the top 10 principal components that capture genetic variation across samples. The results are saved with the prefix pca, producing files like .eigenvec (sample coordinates) and .eigenval (variance explained). This step produces results from  PCA  which reduces high-dimensional SNP data into a few components, allowing visualization and interpretation of population structure and genetic relationships.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env
cd /content/pca_plink

plink \
  --bfile Pigeons_snp-mf01-LDpruned \
  --pca 10 \
  --out pca \
  --allow-extra-chr

In [ ]:
%%bash
cd /content/pca_plink

cat << 'EOF' > pop_names.txt
Niger
Hoggar1
Senegal-P
Hoggar2
Hoggar3
Senegal
fantail
Starling
Scandaroon
Carneau
Jacobin
Lahore
Lebanon
Cumulet
Runt
Englsh_trumpt
Racing
Oriental
Iran1
Algeria1
Kashmir2
Hebrides1
Mykonos
Libya1
Egypt1
Gebeit_Sudan
Athlit
Jerico
Iraq3
Tunisia
Libya2
Algeria2
Ireland
Orkney
Italy2
Ghana
Mali
Kashmir1
Gebeit_Sudan-T
Sudan
Darfur_Sudan1
Mali-T
Darfur_Sudan2
Saudi
Egypt2
Oman
EOF

This step creates an R script (`pca_plot.R`) that visualizes the PCA results generated by PLINK. The script first loads the PCA output files (`pca.eigenvec` and `pca.eigenval`), assigns column names, and calculates the percentage of variance explained by each principal component. It then defines population labels for the samples and combines all information into a single data frame for plotting.

The script uses R packages ggplot2 and ggrepel to generate a scatter plot of PC1 vs PC2. Points are colored by population, and sample names are added as labels without overlapping. The axes are annotated with the percentage of variance explained, making the plot more informative.

Finally, the plot is saved as a PDF file (`Rplots.pdf`) for easy sharing and also printed to the screen for immediate visualization.


In [ ]:
%%bash

cat << 'EOF' > /content/pca_plink/pca_plot.R

pca <- read.table("pca.eigenvec", header = FALSE)
colnames(pca) <- c("FID", "IID", "PC1", "PC2")

eig <- scan("pca.eigenval")
var_exp <- eig / sum(eig) * 100

names <- read.table("pop_names.txt")

pop <- factor(c(rep("Sahara", 2), rep("West Africa", 1), rep("Sahara", 2),
                rep("West Africa", 1), rep("Domestic", 12),
                rep("Middle East", 1), rep("MB-AI", 1), rep("Central Asia", 1),
                rep("MB-AI", 4), rep("Middle East",4), rep("MB-AI",6),
                rep("West Africa", 2), rep("Central Asia", 1),
                rep("Middle East",2), rep("Sahara",1),
                rep("West Africa",1), rep("Sahara",1),
                rep("Middle East",1), rep("MB-AI",1),
                rep("Middle East",1)))

df <- cbind(pca, pop = pop, label = names[,1])

library(tidyverse)
library(ggplot2)
library(ggrepel)

ggplot(df, aes(PC1, PC2, colour = pop)) +
  geom_point(size = 3.5, alpha = 0.6) +
  geom_text_repel(aes(label = label), size = 2.4, segment.size = 0.2,max.overlaps = Inf) +
  theme_minimal() +
  labs(color = "") +
  xlab(paste0("PC1 (", round(var_exp[1], 2), "%)")) +
  ylab(paste0("PC2 (", round(var_exp[2], 2), "%)"))

EOF

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env
cd /content/pca_plink
Rscript pca_plot.R

Run the below cell to visualise the PCA plot.

In [ ]:
!apt-get install -y poppler-utils

!pdftoppm /content/pca_plink/Rplots.pdf Rplot -png

from IPython.display import Image, display

# Display both images
display(Image("Rplot-1.png"))